# PARAMETROS H.DETECTION()

In [ ]:
!pip install herdingspikes==0.3.100 --no-build-isolation

In [ ]:
from datetime import datetime
import sys, os
from herdingspikes.hs2 import HSDetection
from herdingspikes.hs2 import HSClustering
from herdingspikes.probe import BioCam
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pylab

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#Recording file
brw_file = ('/content/drive/Shareddrives/Lab 112/SAMP MEA/samp_2meses_p60 /AN5/SLC1/Phase_01.brw')
print(f'Recording file: {brw_file}')
out_file = brw_file.replace('.brw', '')

#Load BioCam recording
probe = BioCam(brw_file)
print(f'Channels: {probe.num_channels}')

In [ ]:
#Load recording segment for noise inspection
t_start = 1_000_000
t_end = 1_100_000

data = probe.Read(t_start,t_end).reshape(t_end - t_start, probe.num_channels)
print(f'Loaded segment shape: {data.shape}')

#Detect noisy channels based on signal standard deviation
noise_threshold = 15
channel_std = np.std(data, axis=0)
masked_channels = np.where(channel_std > noise_threshold)[0]

print(
    f'Noisy channels detected: '
    f'{len(masked_channels)}')
print(*masked_channels, sep=',')

In [ ]:
threshold = 30 # detection threshold
to_localize = True  # set to False if spikes should only be detected, not localised (will break sorting)
save_all = True
left_cutout_ms = 5.0  #  the number of milliseconds, before the spike, to be included in the spike shape
right_cutout_ms = 5.0  #  the number of milliseconds, after the spike, to be included in the spike shape
amp_evaluation_time = 0.4 # the number of ms during which the trace is integrated, for evaluating amplitude, later comparison with 'maa'
spk_evaluation_time = 1.7

In [ ]:
maskedch = [366,1078,1454,1471,1740,1860,2069,2277,2455,2456,2518,2519,2520,2521,2582,2583,2584,2647,2911,2956,3089,3673]
print(f'Masked Channels: {np.size(maskedch)}, first 10: {maskedch[:10]}')

#Reload BioCam object with masked channels
probe = BioCam(brw_file, masked_channels=maskedch)

In [ ]:
probe.show(show_neighbors=[650], figwidth=28)

# SPIKE DETECTION

In [ ]:
H = HSDetection(probe, num_com_centers=1, threshold=threshold, to_localize=to_localize,
                maa=0, ahpthr=0, out_file_name=out_file, save_all=save_all,
                left_cutout_time=left_cutout_ms, right_cutout_time=right_cutout_ms,
                amp_evaluation_time=amp_evaluation_time, spk_evaluation_time=spk_evaluation_time,
                file_directory_name='analysis', decay_filtering=True)

In [ ]:
H.DetectFromRaw()

In [ ]:
H.LoadDetected()

In [ ]:
H.spikes

In [ ]:
#Raster Plot
fig=plt.figure(figsize=(40, 10))
plt.plot(H.spikes.t/7062.0, H.spikes.ch,'.', color='black')
plt.plot(np.zeros(len(maskedch)),maskedch,'o', color='red')
plt.grid(False)
plt.xlabel('t', fontsize=25)
plt.ylabel('Channels', fontsize=25)
plt.xlim(0, 600)
plt.ylim(0, 4096)
plt.xticks(fontsize=20);
plt.yticks(fontsize=20);

figName = brw_file[0:-4]+'_Raster_original_thr25.jpg'
fig.savefig(figName)

In [ ]:
#Save the DataFrame to CSV
df = pd.read_csv('H.spikes', sep=',', header=0, names=['channel', 'time', 'amplitude', 'x', 'y', 'waveform'])

output_filename = 'df_spikes.csv'
df.to_csv(output_filename, index=False)
print(f'DataFrame saved successfully to: {output_filename}')

In [ ]:
def parse_shape(shape_str):
    shape_str = shape_str.strip("[]")
    return np.array([float(x) for x in shape_str.split()])

channels = df['channel'].unique()
n_channels = len(channels)
n_cols = 5
n_lines = int(np.ceil(n_channels / n_cols))
fig, axs = plt.subplots(n_lines, n_cols, figsize=(15, n_lines * 3))
axs = axs.flatten()

lowactivity_channels = []
for i, channel in enumerate(channels):
    df_channel = df[df['channel'] == channel]
    events = len(df_channel)

    if events < 5:
        lowactivity_channels.append(int(channel))

    #Plot
    for _, row in df_channel.iterrows():
        shape_array = parse_shape(row['waveform'])
        axs[i].plot(shape_array, alpha=0.5)

    axs[i].set_title(f'channel {channel}')
    axs[i].set_xlabel('Channel')
    axs[i].set_ylabel('Amp')

for j in range(i + 1, len(axs)):
    fig.delaxes(axs[j])
plt.tight_layout()
plt.show()

if lowactivity_channels:
    canais_str = ', '.join(map(str, lowactivity_channels))
    print("\nChannels with less than 5 events:")
    print(canais_str)
else:
    print("\nAll good.")